# RSNA Knee — score, blend and package runs (CPU)

**Settings:** Accelerator **None** · Internet **off**.
**Inputs:** datasets `rsna-knee-train-inputs`, `rsna-knee-runs`, `rsna-knee-models` (only for its `wheels/`), and the output of every training notebook whose folds you need (`rsna-knee-train`, …).

1. Set `RUNS` (and optionally `WEIGHTS`) in the next cell → **Run All** (~2 min). Read the fold table and the blend table.
2. To use the result for a submission: **Save Version → Save & Run All**. In the submission notebook, remove `rsna-knee-models` from the inputs and add **this notebook's output** instead (after every new version: Input panel → ⋮ → check for updates).

Folds of one run are merged by run name across all inputs. Generated by `scripts/build_score_notebook.py` — edit the repo, not this notebook.

In [ ]:
import os, sys, time, json, glob, shutil, subprocess, re

RUNS = ["ft2", "ft3a"]      # M2 runs to score and blend (folder names under m2/)
WEIGHTS = None              # None = best blend by OOF auc_llm; or force e.g. {"ft3a": 1.0}
PACK = True                 # write model_dataset/ for the submission notebook

INPUT, WORKING = "/kaggle/input", "/kaggle/working"
CODE, TREE = "/tmp/code", "/tmp/work"

In [ ]:
def find(predicate, top=INPUT, max_depth=8):
    hits = []
    top_depth = top.rstrip("/").count("/")
    for root, dirs, files in os.walk(top, followlinks=True):
        dirs[:] = [d for d in dirs if d not in {"train_series", "test_series", "train_images", "test_images"}]
        if root.count("/") - top_depth >= max_depth:
            dirs[:] = []
        if predicate(root, files):
            hits.append(root)
    return sorted(hits)

def source_name(path):
    # /kaggle/input/datasets/<user>/<slug>/...  or  /kaggle/input/notebooks/<user>/<slug>/...
    parts = os.path.relpath(path, INPUT).split(os.sep)
    return parts[2] if len(parts) > 2 and parts[0] in ("datasets", "notebooks", "models") else parts[0]

LABEL_DIR = find(lambda r, f: "soft_labels.parquet" in f and "folds.csv" in f)
assert LABEL_DIR, "rsna-knee-train-inputs not found - Add Input -> Datasets"
LABEL_DIR = LABEL_DIR[0]
DINO = [f"{d}/model.safetensors" for d in find(lambda r, f: "model.safetensors" in f and r.endswith("dinov2"))]
DINO.sort(key=lambda p: not p.startswith(LABEL_DIR))               # prefer the train-inputs copy
PRIORS = [f"{d}/priors.json" for d in find(lambda r, f: "priors.json" in f)]
PRIORS.sort(key=lambda p: not p.startswith(LABEL_DIR))           # prefer the train-inputs copy
WHEEL_DIR = find(lambda r, f: any(x.endswith(".whl") for x in f))

RUN_DIRS = {}                                                     # run -> [source dirs]
for d in find(lambda r, f: any(re.fullmatch(r"(oof_)?fold\d\.(csv|pt)", x) for x in f)):
    if os.path.basename(d) == "m1" or os.path.basename(d).startswith("m2_"):
        continue                                                  # packaged submission folders
    RUN_DIRS.setdefault(os.path.basename(d), []).append(d)

print("labels :", LABEL_DIR)
print("dinov2 :", DINO[:1] or "NOT FOUND")
print("priors :", PRIORS[:1] or "NOT FOUND")
print("wheels :", WHEEL_DIR[:1] or "NOT FOUND")
print("runs found:")
for run, ds in sorted(RUN_DIRS.items()):
    print(f"  {run:10s} " + " + ".join(source_name(d) for d in ds))
missing = [r for r in RUNS if r not in RUN_DIRS]
assert not missing, f"runs {missing} not in any input - attach the dataset / training notebook that has them"

In [ ]:
# One folder per run under TREE/runs/m2/<run>, with symlinks to every source's files.
IGNORE_CFG = {"root", "name", "folds", "bs", "accum", "amp", "workers", "weights", "log_every",
              "eval_every", "eval_start", "collect", "smoke"}
for d in (f"{CODE}/src/models", f"{TREE}/runs/m2", WORKING):
    os.makedirs(d, exist_ok=True)
for f in ("src/__init__.py", "src/models/__init__.py"):
    open(f"{CODE}/{f}", "w").close()

problems = []
for run in RUNS:
    dst = f"{TREE}/runs/m2/{run}"
    shutil.rmtree(dst, ignore_errors=True); os.makedirs(dst)
    origin, cfgs = {}, []
    for src in RUN_DIRS[run]:
        for f in sorted(os.listdir(src)):
            if f.startswith("config") and f.endswith(".json"):
                cfgs.append((f"{source_name(src)}/{f}", json.load(open(f"{src}/{f}"))))
            if not re.fullmatch(r"(oof_)?fold\d\.(csv|pt)", f):
                continue
            if os.path.exists(f"{dst}/{f}"):
                problems.append(f"{run}: {f} in both {origin[f]} and {source_name(src)} - using {origin[f]}")
                continue
            os.symlink(f"{src}/{f}", f"{dst}/{f}")
            origin[f] = source_name(src)
    pts = {int(f[4]) for f in origin if f.endswith(".pt")}
    oofs = {int(f[8]) for f in origin if f.endswith(".csv")}
    print(f"{run}:")
    for k in range(5):
        src_k = origin.get(f"fold{k}.pt") or origin.get(f"oof_fold{k}.csv") or "-"
        flags = ("ckpt " if k in pts else "NO-CKPT ") + ("oof" if k in oofs else "NO-OOF")
        print(f"   fold {k}: {flags:12s} from {src_k}")
    if pts != set(range(5)) or oofs != set(range(5)):
        problems.append(f"{run}: incomplete - checkpoints {sorted(pts)}, OOF {sorted(oofs)}")
    if cfgs:                                       # settings that must match across folds
        ref_name, ref = cfgs[0]
        for name, c in cfgs[1:]:
            diff = {k: (ref.get(k), c.get(k)) for k in set(ref) | set(c)
                    if k not in IGNORE_CFG and ref.get(k) != c.get(k)}
            if diff:
                problems.append(f"{run}: {name} differs from {ref_name}: {diff}")
        keep = {k: v for k, v in ref.items() if k not in IGNORE_CFG}
        print("   settings:", ", ".join(f"{k}={keep[k]}" for k in
              ("frozen_blocks", "epochs", "img_size", "slices", "lr_backbone", "init") if k in keep))
print("\n" + ("\n".join("WARNING: " + p for p in problems) if problems else "all runs complete and consistent"))

## Source files (embedded from the repo)

In [ ]:
%%writefile /tmp/code/src/models/folds.py
"""5-fold CV split over all training studies, grouped by patient.

    python -m src.models.folds

* Patient grouping: PatientID from data/raw/study_meta_chunk*.csv; all studies of a
  patient go to the same fold. Prints how many patients have more than one study.
* Stratification (greedy, deterministic): the 12 labels (soft label >= 0.5 counts as
  positive) plus the report language, so every fold has a similar mix of findings and
  reporting sites. The 58 human-labelled studies are spread across folds too.

Output: data/labels/folds.csv  StudyInstanceUID, PatientID, fold, lang, source
"""
from __future__ import annotations

import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]

_LANG_HINTS = {
    "es": r"\b(menisco|derrame|rotura|ligamento|rodilla|sin|con|del)\b",
    "de": r"\b(meniskus|kreuzband|erguss|kniegelenk|und|der|die|kein)\b",
    "hr": r"\b(menisk[a-z]*|ligament[a-z]*|zglob[a-z]*|uredn[a-z]*|bez|uz)\b|[čćđ]",
    "nl": r"\b(van|het|een|geen|gewricht[a-z]*|kniegewricht)\b",
    "fr": r"\b(ménisque|épanchement|genou|du|des|pas)\b",
    "tr": r"[ığş]|\b(menisküs|bağ|izlendi|mevcuttur|normaldir)\b",
}


def guess_language(text: str) -> str:
    """Rough report-language guess (used only to balance folds)."""
    t = str(text).lower()
    if re.search(r"[а-яё]", t):
        return "bg"
    if re.search(r"[α-ω]", t):
        return "el"
    scores = {lang: len(re.findall(pat, t)) for lang, pat in _LANG_HINTS.items()}
    lang, best = max(scores.items(), key=lambda kv: kv[1])
    en = len(re.findall(r"\b(the|of|and|is|with|no|meniscus|ligament|effusion)\b", t))
    return lang if best > max(2, en) else "en"


def read_labels(path: Path) -> pd.DataFrame:
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype={"StudyInstanceUID": str})


def load_patients(raw: Path) -> pd.Series | None:
    files = sorted(raw.glob("study_meta_chunk*.csv"))
    if not files:
        return None
    meta = pd.concat([pd.read_csv(f, usecols=["StudyInstanceUID", "PatientID"], dtype=str)
                      for f in files])
    return meta.drop_duplicates("StudyInstanceUID").set_index("StudyInstanceUID")["PatientID"]


def greedy_group_folds(Y: np.ndarray, groups: np.ndarray, k: int, seed: int) -> np.ndarray:
    """Assign each group to a fold, keeping every column's positives and the fold
    sizes as even as possible. Rare columns are placed first."""
    G = groups.max() + 1
    gY = np.zeros((G, Y.shape[1]))
    np.add.at(gY, groups, Y)
    gsize = np.bincount(groups, minlength=G).astype(float)
    totals = Y.sum(0)
    tgt = totals / k
    tgt_size = len(groups) / k
    w = 1.0 / np.maximum(tgt, 1.0)

    rng = np.random.default_rng(seed)
    with np.errstate(divide="ignore"):
        rarity = np.where(gY > 0, totals[None, :], np.inf).min(1)   # rarest column present
    order = np.lexsort((rng.random(G), -gsize, rarity))

    cnt = np.zeros((k, Y.shape[1]))
    size = np.zeros(k)
    fold_of_group = np.empty(G, dtype=int)
    for g in order:
        before = (w * (cnt - tgt) ** 2).sum(1) + ((size - tgt_size) / tgt_size) ** 2 * 50
        after = (w * (cnt + gY[g] - tgt) ** 2).sum(1) + ((size + gsize[g] - tgt_size) / tgt_size) ** 2 * 50
        f = int(np.argmin(after - before))
        fold_of_group[g] = f
        cnt[f] += gY[g]
        size[f] += gsize[g]
    return fold_of_group[groups]


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", type=Path, default=Path("."))
    ap.add_argument("--labels", type=Path, default=None)
    ap.add_argument("--k", type=int, default=5)
    ap.add_argument("--seed", type=int, default=2026)
    args = ap.parse_args()

    raw = args.root / "data" / "raw"
    lab_path = args.labels or args.root / "data" / "labels" / "soft_labels.parquet"
    labels = read_labels(lab_path)
    train = pd.read_csv(raw / "train.csv", usecols=["StudyInstanceUID", "Report"])
    df = labels.merge(train, on="StudyInstanceUID", how="left")
    df = df[df["source"] != "missing"].reset_index(drop=True)

    pids = load_patients(raw)
    if pids is None:
        print("WARNING: no study_meta_chunk*.csv in data/raw -> no patient grouping")
        df["PatientID"] = ""
    else:
        df["PatientID"] = df["StudyInstanceUID"].map(pids).fillna("")
    key = df["PatientID"].where(df["PatientID"] != "", "study:" + df["StudyInstanceUID"])
    groups = pd.factorize(key)[0]
    per_patient = pd.Series(groups).value_counts()
    print(f"studies {len(df)} · patients {groups.max() + 1} · "
          f"patients with >1 study: {(per_patient > 1).sum()} "
          f"(covering {per_patient[per_patient > 1].sum()} studies) · "
          f"missing PatientID: {(df['PatientID'] == '').sum()}")

    df["lang"] = df["Report"].map(guess_language)
    Yl = (df[LABELS].to_numpy() >= 0.5).astype(float)
    Ylang = pd.get_dummies(df["lang"]).to_numpy().astype(float)
    Yhuman = (df["source"] == "human").to_numpy()[:, None].astype(float)
    Y = np.hstack([Yl, Ylang, Yhuman])
    df["fold"] = greedy_group_folds(Y, groups, args.k, args.seed)

    out = args.root / "data" / "labels" / "folds.csv"
    df[["StudyInstanceUID", "PatientID", "fold", "lang", "source"]].to_csv(out, index=False)

    summary = pd.DataFrame(Yl, columns=LABELS).groupby(df["fold"]).sum().astype(int)
    summary.insert(0, "studies", df.groupby("fold").size())
    summary["human"] = df.groupby("fold")["source"].apply(lambda s: (s == "human").sum())
    with pd.option_context("display.width", 200, "display.max_columns", 30):
        print("\npositives (>= 0.5) per fold:")
        print(summary.to_string())
        print("\nlanguages per fold:")
        print(pd.crosstab(df["fold"], df["lang"]).to_string())
    if pids is not None:
        leak = df.groupby("PatientID")["fold"].nunique()
        leak = leak[(leak > 1) & (leak.index != "")]
        print(f"\npatients split across folds: {len(leak)} (must be 0)")
    print(f"\nsaved -> {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /tmp/code/src/models/metrics.py
"""Out-of-fold scoring for the 12 labels.

Two views, reported side by side:
  auc_llm   : LLM-labelled studies; per label the 'uncertain' (0.5) rows are left out
              and the rest are 1/0. Main signal (thousands of studies).
  auc_human : the 58 human-labelled studies (true labels). Small and noisy, but the
              only check against the organisers' own definitions.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]


def safe_auc(y: np.ndarray, s: np.ndarray) -> float:
    y = np.asarray(y)
    if len(y) == 0 or y.min() == y.max():
        return float("nan")
    return float(roc_auc_score(y, s))


def score(pred: np.ndarray, target: np.ndarray, source: np.ndarray) -> pd.DataFrame:
    """pred, target: [n, 12]; source: 'llm' / 'human' per row."""
    rows = []
    llm = source == "llm"
    hum = source == "human"
    for j, lab in enumerate(LABELS):
        t, p = target[:, j], pred[:, j]
        certain = llm & (t != 0.5)
        rows.append(dict(
            label=lab,
            auc_llm=safe_auc(t[certain] >= 0.5, p[certain]),
            pos_llm=int((t[certain] >= 0.5).sum()),
            auc_human=safe_auc(t[hum] >= 0.5, p[hum]),
            pos_human=int((t[hum] >= 0.5).sum()),
        ))
    df = pd.DataFrame(rows).set_index("label")
    df.loc["MACRO"] = [df["auc_llm"].mean(), df["pos_llm"].sum(),
                       df["auc_human"].mean(), df["pos_human"].sum()]
    return df


In [ ]:
%%writefile /tmp/code/src/models/blend.py
"""Choose blend weights for M1 + any number of M2 runs from out-of-fold predictions.

    python -m src.models.blend --m2 ft1,ft2                 # M1 v2 + two M2 runs
    python -m src.models.blend --m1 none --m2 ft1,ft2       # M2 runs only

Scores every model alone, the equal-weight average, and every weighted probability
average on a grid (step 0.05 for up to 3 models, 0.1 for more) on the studies that have
OOF predictions from all models. The best weights by auc_llm (the CV number that has
matched the public LB) go to runs/blend.json, which scripts/make_model_dataset.py reads
to decide which runs go into the Kaggle dataset.

blend.json: {"weights": {"m1": w, "m2_ft1": w, ...}, "runs": {"m1": "v2", "m2_ft1": "ft1", ...},
             "auc_llm": ..., ...}   -- keys are the folder names inside the Kaggle dataset.
"""
from __future__ import annotations

import argparse
import itertools
import json
from pathlib import Path

import numpy as np
import pandas as pd

from .folds import read_labels
from .metrics import LABELS, score


def load_m1(root: Path, run: str) -> pd.DataFrame:
    return pd.read_csv(root / "runs" / "m1" / run / "oof.csv", dtype={"StudyInstanceUID": str})


def load_m2(root: Path, run: str) -> pd.DataFrame:
    parts = sorted((root / "runs" / "m2" / run).glob("oof_fold*.csv"))
    if not parts:
        raise SystemExit(f"no oof_fold*.csv in runs/m2/{run}")
    if len(parts) < 5:
        print(f"WARNING: runs/m2/{run} has only {len(parts)} folds of OOF predictions")
    o = pd.concat([pd.read_csv(p, dtype={"StudyInstanceUID": str}) for p in parts])
    return o.drop_duplicates("StudyInstanceUID", keep="last")


def weight_grid(n: int, step: float):
    k = int(round(1 / step))
    for c in itertools.product(range(k + 1), repeat=n - 1):
        if sum(c) <= k:
            yield np.array([*c, k - sum(c)], float) / k


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", type=Path, default=Path("."))
    ap.add_argument("--m1", default="v2", help="M1 run, or 'none'")
    ap.add_argument("--m2", default="ft1", help="comma-separated M2 runs, e.g. ft1,ft2")
    ap.add_argument("--labels", type=Path, default=None)
    ap.add_argument("--out", type=Path, default=None, help="default runs/blend.json")
    args = ap.parse_args()

    models: dict[str, tuple[str, pd.DataFrame]] = {}       # dataset folder -> (run, oof)
    if args.m1.lower() != "none":
        models["m1"] = (args.m1, load_m1(args.root, args.m1))
    for run in [r.strip() for r in args.m2.split(",") if r.strip()]:
        models[f"m2_{run}"] = (run, load_m2(args.root, run))
    if not models:
        raise SystemExit("no models given")

    labels = read_labels(args.labels or args.root / "data" / "labels" / "soft_labels.parquet")
    df = labels[["StudyInstanceUID", "source", *LABELS]]
    for key, (_, o) in models.items():
        df = df.merge(o[["StudyInstanceUID", *LABELS]].rename(
            columns={l: f"{l}|{key}" for l in LABELS}), on="StudyInstanceUID")
    y, src = df[LABELS].to_numpy(), df["source"].to_numpy()
    keys = list(models)
    P = np.stack([df[[f"{l}|{k}" for l in LABELS]].to_numpy() for k in keys])   # [M, n, 12]
    print(f"{len(df)} studies with OOF from all of: {', '.join(keys)}\n")

    def macro(w):
        sc = score(np.tensordot(w, P, 1), y, src)
        return sc.loc["MACRO", "auc_llm"], sc.loc["MACRO", "auc_human"]

    rows = []
    for i, k in enumerate(keys):
        w = np.eye(len(keys))[i]
        rows.append(dict(blend=f"{k} alone", w=w, **dict(zip(("auc_llm", "auc_human"), macro(w)))))
    if len(keys) > 1:
        w = np.full(len(keys), 1 / len(keys))
        rows.append(dict(blend="equal weights", w=w, **dict(zip(("auc_llm", "auc_human"), macro(w)))))
    step = 0.05 if len(keys) <= 3 else 0.1
    grid = [dict(w=w, auc_llm=macro(w)[0]) for w in weight_grid(len(keys), step)] \
        if len(keys) > 1 else []
    if grid:
        best = max(grid, key=lambda r: r["auc_llm"])
        rows.append(dict(blend="best grid", w=best["w"],
                         **dict(zip(("auc_llm", "auc_human"), macro(best["w"])))))
    tab = pd.DataFrame(rows)
    tab["weights"] = tab["w"].apply(lambda w: " ".join(f"{k}={x:.2f}" for k, x in zip(keys, w)))
    with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 200):
        print(tab[["blend", "auc_llm", "auc_human", "weights"]].to_string(index=False))

    chosen = tab.iloc[-1]
    w = chosen["w"]
    per_label = pd.DataFrame({k: score(P[i], y, src)["auc_llm"] for i, k in enumerate(keys)})
    if len(keys) > 1:
        per_label["blend"] = score(np.tensordot(w, P, 1), y, src)["auc_llm"]
    with pd.option_context("display.float_format", "{:.4f}".format, "display.width", 200):
        print("\nper-label auc_llm:\n" + per_label.to_string())

    out = args.out or args.root / "runs" / "blend.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps({
        "weights": {k: round(float(x), 3) for k, x in zip(keys, w)},
        "runs": {k: run for k, (run, _) in models.items()},
        "auc_llm": round(float(chosen["auc_llm"]), 4),
        "auc_human": round(float(chosen["auc_human"]), 4),
        "n_studies": int(len(df))}, indent=2))
    print(f"\nchosen: {chosen['weights']}  ->  auc_llm {chosen['auc_llm']:.4f}")
    print(f"saved -> {out}")


if __name__ == "__main__":
    main()


## Score each run, fold by fold

In [ ]:
sys.path.insert(0, CODE)
from pathlib import Path
import numpy as np, pandas as pd
from src.models.folds import read_labels
from src.models.metrics import LABELS, score

lab = read_labels(Path(f"{LABEL_DIR}/soft_labels.parquet"))[["StudyInstanceUID", "source", *LABELS]]

def run_score(parts):
    o = pd.concat([pd.read_csv(p, dtype={"StudyInstanceUID": str}) for p in parts])
    df = lab.merge(o, on="StudyInstanceUID", suffixes=("", "_p"))
    return score(df[[f"{l}_p" for l in LABELS]].to_numpy(), df[LABELS].to_numpy(),
                 df["source"].to_numpy()), len(df)

table, per_label, counts = {}, {}, {}
for run in RUNS:
    parts = sorted(glob.glob(f"{TREE}/runs/m2/{run}/oof_fold*.csv"))
    row = {}
    for p in parts:
        sc, _ = run_score([p])
        row[f"fold {p[-5]}"] = sc.loc["MACRO", "auc_llm"]
    sc, n = run_score(parts)
    row["ALL (OOF)"], row["auc_human"] = sc.loc["MACRO", "auc_llm"], sc.loc["MACRO", "auc_human"]
    table[run], per_label[run], counts[run] = row, sc["auc_llm"], n
pd.set_option("display.width", 200)
print("macro auc_llm per fold (auc_human = 58 human studies, noisy):")
print(pd.DataFrame(table).T.to_string(float_format="{:.4f}".format))
print("studies scored:", counts)
pd.DataFrame(table).T.to_csv(f"{WORKING}/scores.csv")

## Blend

In [ ]:
os.makedirs(f"{WORKING}/model_dataset", exist_ok=True)
BLEND = f"{WORKING}/model_dataset/blend.json"
r = subprocess.run([sys.executable, "-m", "src.models.blend", "--root", TREE, "--m1", "none",
                    "--m2", ",".join(RUNS), "--labels", f"{LABEL_DIR}/soft_labels.parquet",
                    "--out", BLEND], cwd=CODE, capture_output=True, text=True)
print(r.stdout[-6000:])
assert r.returncode == 0, r.stderr[-3000:]

if WEIGHTS:                                        # forced weights instead of the best grid
    b = json.load(open(BLEND))
    keys = [f"m2_{run}" for run in RUNS]
    w = np.array([float(WEIGHTS.get(run, 0.0)) for run in RUNS]); w = w / w.sum()
    oofs = []
    for run in RUNS:
        o = pd.concat([pd.read_csv(p, dtype={"StudyInstanceUID": str})
                       for p in sorted(glob.glob(f"{TREE}/runs/m2/{run}/oof_fold*.csv"))])
        oofs.append(o.drop_duplicates("StudyInstanceUID", keep="last").set_index("StudyInstanceUID"))
    common = sorted(set.intersection(*(set(o.index) for o in oofs)))
    P = sum(wi * o.loc[common, LABELS].to_numpy() for wi, o in zip(w, oofs))
    df = lab.set_index("StudyInstanceUID").loc[common]
    sc = score(P, df[LABELS].to_numpy(), df["source"].to_numpy())
    b.update(weights={k: round(float(x), 3) for k, x in zip(keys, w)},
             auc_llm=round(float(sc.loc["MACRO", "auc_llm"]), 4),
             auc_human=round(float(sc.loc["MACRO", "auc_human"]), 4), n_studies=len(common))
    json.dump(b, open(BLEND, "w"), indent=2)
    print(f"FORCED weights {b['weights']} -> auc_llm {b['auc_llm']:.4f}, auc_human {b['auc_human']:.4f}")
print("blend.json:", json.dumps(json.load(open(BLEND))))

## Package for the submission notebook

In [ ]:
MD = f"{WORKING}/model_dataset"
if PACK:
    assert DINO, "no dinov2/model.safetensors in the inputs (rsna-knee-train-inputs or rsna-knee-models)"
    assert PRIORS, "no priors.json in the inputs"
    assert WHEEL_DIR, "no wheels - attach rsna-knee-models (its wheels/ folder has the DICOM decoders)"
    b = json.load(open(BLEND))
    used = {k: w for k, w in b["weights"].items() if w > 0}
    for key, w in used.items():
        run = b["runs"][key]
        os.makedirs(f"{MD}/{key}", exist_ok=True)
        for f in sorted(os.listdir(f"{TREE}/runs/m2/{run}")):
            if f.endswith(".pt"):
                shutil.copy(os.path.realpath(f"{TREE}/runs/m2/{run}/{f}"), f"{MD}/{key}/{f}")
        for src in RUN_DIRS[run]:                     # settings, for reference only
            for f in glob.glob(f"{src}/config*.json"):
                shutil.copy(f, f"{MD}/{key}/{source_name(src)}_{os.path.basename(f)}")
        print(f"{key:12s} weight {w}: {len(glob.glob(f'{MD}/{key}/fold*.pt'))} folds")
    dropped = [k for k, w in b["weights"].items() if w <= 0]
    if dropped:
        print("weight 0, left out:", ", ".join(dropped))
    json.dump({**b, "weights": used}, open(BLEND, "w"), indent=2)
    os.makedirs(f"{MD}/dinov2", exist_ok=True)
    shutil.copy(DINO[0], f"{MD}/dinov2/model.safetensors")
    shutil.copy(PRIORS[0], f"{MD}/priors.json")
    shutil.copytree(WHEEL_DIR[0], f"{MD}/wheels", dirs_exist_ok=True)
    for root, _, files in sorted(os.walk(MD)):
        for f in sorted(files):
            p = os.path.join(root, f)
            print(f"  {os.path.relpath(p, MD):42s} {os.path.getsize(p) / 1e6:8.1f} MB")
    total = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(MD) for f in fs) / 1e9
    print(f"model_dataset: {total:.2f} GB -> Save Version, then attach this notebook's output "
          f"to the submission notebook")
else:
    shutil.rmtree(MD, ignore_errors=True)